# Recipe Data Collection

Collect publicly accessible Lithuanian recipes from the
Jaukūs skoniai API and preserve raw JSON responses
for subsequent data cleaning and normalisation.

## 1. Setup and Imports

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from scrapers import recipes_jaukus
from scrapers.common import save_rows

## 2. Recipe Catalogue Collection

In [ ]:
catalogue = recipes_jaukus.fetch_catalogue(max_pages=10)

df_catalogue = pd.DataFrame(catalogue)

print("Total recipes:", len(df_catalogue))
print("Unique recipes:", df_catalogue["id"].nunique())

In [ ]:
df_public = (
    df_catalogue[df_catalogue["subscriberOnly"].eq(False)]
    .drop_duplicates(subset="id")
    .copy()
)

print("Public recipes:", len(df_public))

In [ ]:
slugs = df_public["slug"].tolist()

slugs[:10]

## 3. Individual Recipe Collection

In [ ]:
recipes_remaining, failed = recipes_jaukus.collect_recipes(
    slugs[3:],
    delay=2
)

recipes = recipes_test + recipes_remaining

In [ ]:
print("Requested:", len(slugs))
print("Collected:", len(recipes))
print("Failed:", len(failed))
print("Not collected:", len(slugs) - len(recipes))
print("Unique recipes:", len({r["id"] for r in recipes}))

In [ ]:
print("Recipes without ingredients:",
      sum(not r["content"].get("ingredients") for r in recipes))

print("Recipes without servings:",
      sum(not r["content"].get("servings") for r in recipes))

print("Recipes without nutrition:",
      sum(not r["content"].get("nutrition") for r in recipes))

## 4. Raw Data Export

Save the collected recipe catalogue and individual recipe
responses as JSON files for subsequent exploratory analysis,
cleaning and normalisation.

In [ ]:
if recipes and len(recipes) == len(slugs) and not failed:
    save_rows(catalogue, "recipes_jaukus", "catalogue")
    save_rows(recipes, "recipes_jaukus", "recipes")
else:
    print("Collection incomplete. Inspect results before saving.")